# 04 · Report: tables and figures

| Setting | Value |
|---|---|
| Accelerator | **None (CPU)** |
| Internet | OFF is fine (ON only if you use `REPO_URL`) |
| Attach | code (`scopesum-code`) unless `REPO_URL`; the output of `03_eval` (folder with `A.json` … `E.json`) |
| Expected runtime | < 1 min |

Produces: main results table (A–E, CSV + LaTeX), B-vs-C bar chart, memory chart, all in `/kaggle/working/report`.

In [ ]:
EVAL_DIR = "/kaggle/input/scopesum-eval/scopesum_eval"   # VERIFY: folder with A.json ... E.json

REPO_URL = "https://github.com/loaferct/scopesum.git"  # private repo: needs the Kaggle Secret GITHUB_TOKEN; "" = use an attached code dataset
CODE_DATASET_DIR = "/kaggle/input"
import glob, os, shutil, subprocess, sys
CODE_DIR = "/kaggle/working/scopesum"

def clone_repo(url, dest, secret_name="GITHUB_TOKEN"):
    """git clone; uses a GitHub token from Kaggle Secrets for private repos (never printed)."""
    auth_url = url
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(secret_name)
        auth_url = url.replace("https://", f"https://x-access-token:{token}@")
    except Exception as e:
        print(f"No Kaggle Secret {secret_name!r} ({type(e).__name__}); trying an anonymous clone")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, dest], capture_output=True, text=True)
    if r.returncode != 0:
        print("git clone failed:", r.stderr.replace(auth_url, url)[-500:])
        return False
    subprocess.run(["git", "-C", dest, "remote", "set-url", "origin", url])  # drop the token from .git/config
    print("cloned", url, subprocess.run(["git", "-C", dest, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
    return True

if not os.path.exists(os.path.join(CODE_DIR, "scopesum", "__init__.py")):
    if not (REPO_URL and clone_repo(REPO_URL, CODE_DIR)):
        hits = [h for h in glob.glob(CODE_DATASET_DIR + "/**/scopesum/__init__.py", recursive=True) if "/tests/" not in h]
        assert hits, "Attach the scopesum code dataset or set REPO_URL"
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE_DIR)
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
if not glob.glob(os.path.join(EVAL_DIR, "*.json")):
    found = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/C.json", recursive=True)})
    print("EVAL_DIR has no results; candidates:", found)
    EVAL_DIR = found[0]

In [ ]:
import pandas as pd
from scopesum.report import collect_results, plot_b_vs_c, plot_memory, results_tables, to_latex

OUT = "/kaggle/working/report"
os.makedirs(OUT, exist_ok=True)
allres = collect_results(EVAL_DIR)
print("variants found:", list(allres))
tables = results_tables(allres)
pd.set_option("display.width", 200)
for name, df in tables.items():
    df.to_csv(os.path.join(OUT, f"{name}.csv"), index=False)
    print(f"\n== {name} ==")
    display(df)
main_cols = [c for c in ["variant", "loss_all", "loss_long_range", "loc_top1", "loc_top5", "kv_mb_mean", "probe_returns_value", "probe_n_args_bucket", "probe_name_reused"] if c in tables["main_results"]]
latex = to_latex(tables["main_results"][main_cols])
open(os.path.join(OUT, "main_table.tex"), "w").write(latex)
print(latex)

In [ ]:
fig = plot_b_vs_c(allres, os.path.join(OUT, "b_vs_c.png"))
fig2 = plot_memory(allres, os.path.join(OUT, "memory.png"))
print(os.listdir(OUT))

In [ ]:
# Fairness reminder for the paper: B and C must have (nearly) equal <sum> counts.
stats_files = glob.glob("/kaggle/input/**/stats.json", recursive=True)
if stats_files:
    import json
    s = json.load(open(stats_files[0]))
    print({k: s[k] for k in ["chunk_size", "scope_sums_total", "fixed_sums_total", "relative_count_gap", "avg_scope_len"]})